In [ ]:
"""
MNIST CNN Baseline - Project Milestone 1
==========================================
Task:    Train a standard (non-equivariant) CNN on MNIST and evaluate on
         both the standard test set and a rotated test set.
Purpose: Provide a baseline for comparison against equivariant networks,
         demonstrating that standard CNNs are not rotation-invariant.

How to run:
    1. Open a new notebook in Google Colab (recommended, free GPU).
    2. Paste the entire file content into a cell and run.
    3. CPU is fine (~10 min); GPU takes ~2-3 min.

Expected results:
    - Standard test set (no rotation):  ~99% accuracy
    - Rotated test set (0-360 deg):     ~40-55% accuracy
      (This drop motivates the need for equivariant architectures.)

AI Usage Disclosure
Claude (Anthropic) was used during Week 2 to support the implementation of the baseline CNN. Specifically:

Code generation: Claude produced an initial version of the training and evaluation script based on the CNN class in Section 2.3 of the UvA tutorial.
Conceptual clarification: Claude was used to walk through every line of the script and explain concepts (e.g., normalization, learning rate, epochs, .to(device), self, class inheritance, __init__, attribute access via self.) in detail, since I had limited prior Python and PyTorch experience.
Verification: I read through every line of the generated code, asked clarifying questions until I understood it, and can defend the architectural and implementation choices.

"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

# -------------------------------
# 1. Configuration
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BATCH_SIZE = 64
EPOCHS = 5         # MNIST is simple; 5 epochs are enough to reach ~99%.
LR = 1e-3

# -------------------------------
# 2. Data: training set + two test sets
# -------------------------------
# Training set: standard MNIST (no rotation augmentation).
# We deliberately do NOT augment with rotations during training, so that
# the baseline CNN's lack of rotation invariance becomes visible.
train_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

# Test set 1: standard (no rotation).
test_tf_normal = train_tf

# Test set 2: random rotation in [0, 360] degrees.
# This is the key controlled comparison.
test_tf_rotated = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation([0, 360], interpolation=transforms.InterpolationMode.BILINEAR, fill=0),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds        = torchvision.datasets.MNIST(root="./data", train=True,  download=True, transform=train_tf)
test_ds_normal  = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=test_tf_normal)
test_ds_rotated = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=test_tf_rotated)

train_loader        = DataLoader(train_ds,        batch_size=BATCH_SIZE, shuffle=True)
test_loader_normal  = DataLoader(test_ds_normal,  batch_size=BATCH_SIZE, shuffle=False)
test_loader_rotated = DataLoader(test_ds_rotated, batch_size=BATCH_SIZE, shuffle=False)


# -------------------------------
# 3. Model: standard CNN
# -------------------------------
# This architecture intentionally mirrors the style of the CNN class in
# Section 2.3 of the UvA tutorial, so that the baseline is a fair
# comparison point for the equivariant networks to follow.
class CNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=10, hidden_channels=32, num_hidden=4, kernel_size=5):
        super().__init__()
        self.first_conv = nn.Conv2d(in_channels, hidden_channels, kernel_size=kernel_size, padding=0)
        self.convs = nn.ModuleList([
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=kernel_size, padding=0)
            for _ in range(num_hidden)
        ])
        self.final_linear = nn.Linear(hidden_channels, num_classes)

    def forward(self, x):
        x = self.first_conv(x)
        x = F.layer_norm(x, x.shape[-3:])
        x = F.relu(x)
        for conv in self.convs:
            x = conv(x)
            x = F.layer_norm(x, x.shape[-3:])
            x = F.relu(x)
        x = F.adaptive_avg_pool2d(x, 1).squeeze(-1).squeeze(-1)  # global average pooling
        return self.final_linear(x)


# -------------------------------
# 4. Training & evaluation
# -------------------------------
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total


model = CNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {num_params:,}\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    acc_normal  = evaluate(model, test_loader_normal)
    acc_rotated = evaluate(model, test_loader_rotated)
    print(f"Epoch {epoch}/{EPOCHS} | loss={total_loss/len(train_loader):.4f} | "
          f"acc(normal)={acc_normal*100:.2f}% | acc(rotated)={acc_rotated*100:.2f}%")

# -------------------------------
# 5. Final results summary
# -------------------------------
print("\n" + "="*55)
print("Final results (Baseline CNN, no equivariant structure)")
print("="*55)
print(f"Number of parameters:        {num_params:,}")
print(f"Standard MNIST test set:     {acc_normal*100:.2f}%")
print(f"Rotated MNIST test set:      {acc_rotated*100:.2f}%")
print(f"Accuracy drop:               {(acc_normal - acc_rotated)*100:.2f} percentage points")
print("="*55)
print("\nConclusion: the standard CNN's accuracy drops substantially on the")
print("rotated test set, motivating the use of equivariant architectures")
print("(e.g., C4/C8 group-equivariant CNNs).")

Using device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 20.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 494kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.64MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.6MB/s]


Number of parameters: 103,690

Epoch 1/5 | loss=0.3864 | acc(normal)=98.43% | acc(rotated)=40.20%
Epoch 2/5 | loss=0.0678 | acc(normal)=99.03% | acc(rotated)=43.17%
Epoch 3/5 | loss=0.0504 | acc(normal)=98.83% | acc(rotated)=43.22%
Epoch 4/5 | loss=0.0418 | acc(normal)=99.12% | acc(rotated)=43.18%
Epoch 5/5 | loss=0.0361 | acc(normal)=99.09% | acc(rotated)=43.69%

Final results (Baseline CNN, no equivariant structure)
Number of parameters:        103,690
Standard MNIST test set:     99.09%
Rotated MNIST test set:      43.69%
Accuracy drop:               55.40 percentage points

Conclusion: the standard CNN's accuracy drops substantially on the
rotated test set, motivating the use of equivariant architectures
(e.g., C4/C8 group-equivariant CNNs).
